# 
YOLOv8 Object Detection with Ultralytics

This notebook demonstrates comprehensive object detection using **YOLOv8** from Ultralytics.

#Contents
1. Setup & Installation
2. Model Initialization
3. Single Image Detection
4. Multi-Image Detection
5. Statistical Analysis
6. Performance Benchmarking
7. Advanced Experiments
8. Export & Reporting

---
## Part 1: Setup & Installation
---

In [ ]:
# Install dependencies
!pip install ultralytics matplotlib seaborn pandas tqdm requests pillow -q

print("✓ Dependencies installed!")

In [ ]:
# Import libraries
import os
import time
import json
from collections import Counter
from io import BytesIO

import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from PIL import Image
import requests
from tqdm import tqdm
import ultralytics
from ultralytics import YOLO

print(f"✓ Libraries imported!")
print(f"  PyTorch: {torch.__version__}")
print(f"  Ultralytics: {ultralytics.__version__}")

In [ ]:
# Check hardware
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA: {torch.version.cuda}")

---
## Part 2: Model Initialization
---

In [ ]:
# Load YOLOv8 model
print("Loading YOLOv8n model...")
model = YOLO('yolov8n.pt')

print(f"\n✓ Model loaded!")
print(f"  Type: YOLOv8n (Nano)")
print(f"  Classes: {len(model.names)}")

# Warm up model
print("\nWarming up model...")
dummy = np.zeros((640, 640, 3), dtype=np.uint8)
_ = model(dummy, verbose=False)
print("✓ Model ready!")

---
## Part 3: Sample Data Loading
---

In [ ]:
# Sample image URLs
SAMPLE_URLS = {
    'street': 'https://ultralytics.com/images/bus.jpg',
    'people': 'https://ultralytics.com/images/zidane.jpg',
}

def load_image_from_url(url):
    """Load image from URL."""
    response = requests.get(url, timeout=10)
    img = Image.open(BytesIO(response.content))
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

# Load sample images
print("Loading sample images...")
sample_images = {}
for name, url in SAMPLE_URLS.items():
    try:
        sample_images[name] = load_image_from_url(url)
        print(f"  ✓ {name}: {sample_images[name].shape}")
    except Exception as e:
        print(f"  ✗ {name}: {e}")

print(f"\n✓ Loaded {len(sample_images)} images!")

---
## Part 4: Single Image Detection
---

In [ ]:
# Detect on single image
demo_name = 'street'
demo_image = sample_images[demo_name]

print(f"Running detection on '{demo_name}'...")
start = time.time()
results = model(demo_image, verbose=False)
inference_time = (time.time() - start) * 1000

print(f"\n✓ Detection complete!")
print(f"  Time: {inference_time:.2f} ms")
print(f"  Detections: {len(results[0].boxes)}")

# Show detected classes
print("\nDetected objects:")
for box in results[0].boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    print(f"  - {model.names[cls_id]}: {conf:.2f}")

In [ ]:
# Visualize results
annotated = results[0].plot()
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 10))
plt.imshow(annotated_rgb)
plt.title(f"YOLOv8 Detection - {demo_name.title()}")
plt.axis('off')
plt.tight_layout()
plt.show()

---
## Part 5: Multi-Image Detection
---

In [ ]:
# Detect on all images
print("Running detection on all images...\n")
all_results = {}

for name, img in sample_images.items():
    start = time.time()
    results = model(img, verbose=False)
    inf_time = (time.time() - start) * 1000
    all_results[name] = results
    n_det = len(results[0].boxes)
    print(f"  {name}: {n_det} detections ({inf_time:.1f} ms)")

print(f"\n✓ Processed {len(all_results)} images!")

In [ ]:
# Display grid of results
n_images = len(sample_images)
fig, axes = plt.subplots(1, n_images, figsize=(7*n_images, 7))

if n_images == 1:
    axes = [axes]

for ax, (name, results) in zip(axes, all_results.items()):
    annotated = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    ax.set_title(f"{name.title()} ({len(results[0].boxes)} objects)")
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## Part 6: Statistical Analysis
---

In [ ]:
# Collect statistics
all_classes = []
all_confidences = []

for name, results in all_results.items():
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        all_classes.append(model.names[cls_id])
        all_confidences.append(conf)

print(f"Total detections: {len(all_classes)}")
print(f"Unique classes: {len(set(all_classes))}")
print(f"\nClass distribution:")
for cls, count in Counter(all_classes).most_common(10):
    print(f"  {cls}: {count}")

print(f"\nConfidence statistics:")
print(f"  Mean: {np.mean(all_confidences):.3f}")
print(f"  Std: {np.std(all_confidences):.3f}")
print(f"  Min: {np.min(all_confidences):.3f}")
print(f"  Max: {np.max(all_confidences):.3f}")

In [ ]:
# Visualize statistics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
class_counts = Counter(all_classes)
classes = list(class_counts.keys())
counts = list(class_counts.values())

axes[0].barh(classes, counts, color='steelblue')
axes[0].set_xlabel('Count')
axes[0].set_title('Detected Classes')
axes[0].invert_yaxis()

# Confidence distribution
axes[1].hist(all_confidences, bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[1].axvline(np.mean(all_confidences), color='red', linestyle='--', 
               label=f'Mean: {np.mean(all_confidences):.3f}')
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Count')
axes[1].set_title('Confidence Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Part 7: Performance Benchmarking
---

In [ ]:
# Benchmark at different image sizes
sizes = [320, 480, 640, 960]
n_runs = 10
benchmark_results = []

print("Running benchmarks...\n")
for size in sizes:
    test_img = np.random.randint(0, 255, (size, size, 3), dtype=np.uint8)
    
    # Warmup
    _ = model(test_img, verbose=False)
    
    # Benchmark
    times = []
    for _ in range(n_runs):
        start = time.time()
        _ = model(test_img, verbose=False)
        times.append((time.time() - start) * 1000)
    
    avg_time = np.mean(times)
    fps = 1000 / avg_time
    
    benchmark_results.append({
        'size': f'{size}x{size}',
        'avg_time_ms': avg_time,
        'fps': fps
    })
    
    print(f"  {size}x{size}: {avg_time:.2f} ms ({fps:.1f} FPS)")

print("\n✓ Benchmark complete!")

In [ ]:
# Visualize benchmark results
sizes_str = [r['size'] for r in benchmark_results]
fps_vals = [r['fps'] for r in benchmark_results]

plt.figure(figsize=(10, 5))
bars = plt.bar(sizes_str, fps_vals, color=['green' if f > 30 else 'orange' if f > 15 else 'red' for f in fps_vals])
plt.axhline(30, color='gray', linestyle='--', label='30 FPS (Real-time)')
plt.xlabel('Image Size')
plt.ylabel('FPS')
plt.title('YOLOv8n Performance Benchmark')
plt.legend()

for bar, fps in zip(bars, fps_vals):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
            f'{fps:.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

---
## Part 8: Export & Reporting
---

In [ ]:
# Generate experiment report
report = {
    'experiment_info': {
        'date': time.strftime('%Y-%m-%d %H:%M:%S'),
        'device': device,
        'model': 'YOLOv8n',
        'ultralytics_version': ultralytics.__version__,
        'pytorch_version': torch.__version__,
        'num_images': len(sample_images)
    },
    'detection_summary': {
        'total_detections': len(all_classes),
        'unique_classes': len(set(all_classes)),
        'avg_confidence': float(np.mean(all_confidences)),
    },
    'top_classes': dict(Counter(all_classes).most_common(10)),
    'performance': {
        size: {'fps': r['fps'], 'ms': r['avg_time_ms']}
        for size, r in zip(sizes_str, benchmark_results)
    }
}

print(json.dumps(report, indent=2))

In [ ]:
# Export detections to CSV
export_data = []
for name, results in all_results.items():
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        bbox = box.xyxy[0].tolist()
        
        export_data.append({
            'image': name,
            'class_name': model.names[cls_id],
            'class_id': cls_id,
            'confidence': conf,
            'x1': bbox[0],
            'y1': bbox[1],
            'x2': bbox[2],
            'y2': bbox[3],
            'center_x': (bbox[0] + bbox[2]) / 2,
            'center_y': (bbox[1] + bbox[3]) / 2,
            'area': (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
        })

df_export = pd.DataFrame(export_data)
print(f"Exported {len(df_export)} detections")
df_export.head(10)

---
## Summary

This notebook demonstrated:
- ✅ YOLOv8 model loading with Ultralytics
- ✅ Single and multi-image detection
- ✅ Statistical analysis of detections
- ✅ Performance benchmarking
- ✅ Data export and reporting

### Next Steps
1. Try different model sizes (yolov8s, yolov8m, yolov8l, yolov8x)
2. Fine-tune on custom datasets
3. Export to ONNX/TensorRT for deployment
4. Integrate with the API for production use

### Resources
- [Ultralytics Docs](https://docs.ultralytics.com)
- [YOLOv8 GitHub](https://github.com/ultralytics/ultralytics)
---